# NovaCart E-commerce Analytics
## Notebook 01 — Data Quality Audit

**Goal:** Inspect all raw client-style datasets before cleaning or transformation.

We will check:

- File structure and row counts
- Column names and data types
- Missing values
- Duplicate records
- Invalid dates
- Invalid numeric values
- Inconsistent text categories
- Primary-key uniqueness
- Foreign-key integrity
- Cross-table business-rule violations

> Important: This notebook performs **audit only**.  
> Do not overwrite any files in `data/raw/`.

## 1. Import libraries

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Define project paths

This notebook assumes the following project structure:

```text
Ecommerce_Analytics_Project/
│
├── data/
│   ├── raw/
│   └── cleaned/
│
└── notebooks/
    └── 01_data_quality.ipynb
```

In [23]:
PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned"

print("Raw data folder:", RAW_DIR.resolve())
print("Clean data folder:", CLEAN_DIR.resolve())

Raw data folder: C:\Users\Didar\Desktop\Ecommerce_Analytics_Project\data\raw
Clean data folder: C:\Users\Didar\Desktop\Ecommerce_Analytics_Project\data\cleaned


## 3. Load all raw datasets

In [24]:
customers = pd.read_csv(RAW_DIR / "customers.csv")
products = pd.read_csv(RAW_DIR / "products.csv")
orders = pd.read_csv(RAW_DIR / "orders.csv")
order_items = pd.read_csv(RAW_DIR / "order_items.csv")
returns = pd.read_csv(RAW_DIR / "returns.csv")
marketing = pd.read_csv(RAW_DIR / "marketing.csv")

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "returns": returns,
    "marketing": marketing
}

print("Files loaded successfully.")

Files loaded successfully.


## 4. Dataset overview

First inspect the number of rows and columns in each file.

In [25]:
overview = []

for name, df in datasets.items():
    overview.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1]
    })

overview_df = pd.DataFrame(overview)
overview_df

,Dataset,Rows,Columns
0,customers,25000,7
1,products,1200,9
2,orders,112000,8
3,order_items,227961,11
4,returns,16185,8
5,marketing,360,10


## 5. Preview each dataset

In [26]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)
    display(df.head())


CUSTOMERS


,Customer_ID,Customer_Name,City,State,Customer_Segment,Signup_Date,Email_Opt_In
0,C000001,Robert Lopez,Austin,TX,Consumer,2025-10-17,Yes
1,C000002,James Gonzalez,Charlotte,NC,Consumer,2025-12-06,Yes
2,C000003,Matthew Martinez,Dallas,TX,Professional,2023-10-27,No
3,C000004,Linda Jones,Other,Other,Consumer,2024-03-02,No
4,C000005,Daniel Rodriguez,New York,NY,Consumer,2023-10-20,Yes



PRODUCTS


,Product_ID,Product_Name,Category,Subcategory,Brand,List_Price,Unit_Cost,Base_Margin,Base_Return_Rate
0,P00001,Brivon Classic Fitness,Sports & Outdoors,Fitness,Brivon,34.88,19.97,0.43,0.07
1,P00002,Mirello Comfort Bags,Fashion,Bags,Mirello,40.70,25.99,0.36,0.13
2,P00003,Solara Pro Stationery,Office,Stationery,Solara,34.21,24.10,0.30,0.04
3,P00004,Nexora Classic Yoga,Sports & Outdoors,Yoga,Nexora,25.01,15.59,0.38,0.07
4,P00005,Orbito Lite Power Banks,Electronics,Power Banks,Orbito,100.96,77.70,0.23,0.09



ORDERS


...

,Order_Item_ID,Order_ID,Product_ID,Quantity,Unit_Price,Discount_Pct,Gross_Sales,Discount_Amount,Net_Sales,Cost_Amount,Profit
0,OI00000001,O0000001,P01120,1,22.55,0.10,22.55,2.26,20.30,13.95,6.35
1,OI00000002,O0000001,P00781,1,26.91,0.00,26.91,0.00,26.91,15.87,11.04
2,OI00000003,O0000002,P00324,1,55.49,0.10,55.49,5.55,49.94,30.60,19.34
3,OI00000004,O0000002,P00283,4,156.92,0.10,627.68,62.77,564.91,413.32,151.59
4,OI00000005,O0000002,P00318,1,54.68,0.05,54.68,2.73,51.95,38.04,13.91



RETURNS


,Return_ID,Order_Item_ID,Order_ID,Product_ID,Return_Date,Return_Reason,Refund_Amount,Return_Status
0,R000001,OI00000006,O0000003,P00318,2024-01-06,Changed Mind,109.36,Pending
1,R000002,OI00000079,O0000034,P00467,2024-01-09,Damaged,11.37,Refunded
2,R000003,OI00000086,O0000036,P00187,2024-01-09,Wrong Size,108.70,Refunded
3,R000004,OI00000087,O0000037,P01183,2024-01-04,Compatibility Issue,363.39,Refunded
4,R000005,OI00000091,O0000039,P00958,2024-01-19,Changed Mind,292.00,Refunded



MARKETING


,Campaign_ID,Month,Channel,Campaign_Type,Spend,Impressions,Clicks,Attributed_Orders,CTR,Conversion_Rate
0,M00001,2024-01-01,Paid Search,Always On,"23,762.47",742577,"28,098.00",1186,0.04,0.04
1,M00002,2024-01-01,Paid Search,Promotion,"13,429.63",419675,"16,605.00",727,0.04,0.04
2,M00003,2024-01-01,Paid Search,Retargeting,"9,774.27",305445,"12,393.00",517,0.04,0.04
3,M00004,2024-01-01,Social Media,Always On,"16,191.51",1471955,"28,705.00",616,0.02,0.02
4,M00005,2024-01-01,Social Media,Promotion,"11,320.95",1029177,"16,799.00",441,0.02,0.03


## 6. Column names and data types

This helps identify whether numeric/date fields were imported incorrectly.

In [27]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)
    print(df.dtypes)


CUSTOMERS
Customer_ID         str
Customer_Name       str
City                str
State               str
Customer_Segment    str
Signup_Date         str
Email_Opt_In        str
dtype: object

PRODUCTS
Product_ID              str
Product_Name            str
Category                str
Subcategory             str
Brand                   str
List_Price          float64
Unit_Cost           float64
Base_Margin         float64
Base_Return_Rate    float64
dtype: object

ORDERS
Order_ID           str
Customer_ID        str
Order_Date         str
Channel            str
Payment_Method     str
Shipping_Method    str
Order_Status       str
Ship_Date          str
dtype: object

ORDER_ITEMS
Order_Item_ID          str
Order_ID               str
Product_ID             str
Quantity             int64
Unit_Price         float64
Discount_Pct       float64
Gross_Sales        float64
Discount_Amount    float64
Net_Sales          float64
Cost_Amount        float64
Profit             float64
dtype: object



## 7. Missing-value audit

In [28]:
missing_report = []

for name, df in datasets.items():
    for col in df.columns:
        missing_count = df[col].isna().sum()
        missing_pct = (missing_count / len(df)) * 100

        if missing_count > 0:
            missing_report.append({
                "Dataset": name,
                "Column": col,
                "Missing_Count": missing_count,
                "Missing_%": round(missing_pct, 2)
            })

missing_df = pd.DataFrame(missing_report)

if len(missing_df):
    display(missing_df.sort_values(["Dataset", "Missing_Count"], ascending=[True, False]))
else:
    print("No missing values found.")

,Dataset,Column,Missing_Count,Missing_%
0,customers,Email_Opt_In,150,0.60
8,marketing,Spend,10,2.78
9,marketing,Clicks,10,2.78
5,order_items,Discount_Pct,300,0.13
4,order_items,Product_ID,150,0.07
3,orders,Ship_Date,4645,4.15
2,orders,Shipping_Method,250,0.22
1,products,Brand,15,1.25
6,returns,Refund_Amount,120,0.74
7,returns,Return_Status,120,0.74


## 8. Duplicate-row audit

We check exact duplicate records in each dataset.

In [29]:
duplicate_report = []

for name, df in datasets.items():
    duplicate_report.append({
        "Dataset": name,
        "Exact_Duplicate_Rows": int(df.duplicated().sum())
    })

duplicate_df = pd.DataFrame(duplicate_report)
duplicate_df

,Dataset,Exact_Duplicate_Rows
0,customers,0
1,products,0
2,orders,0
3,order_items,450
4,returns,0
5,marketing,0


## 9. Primary-key uniqueness

Expected unique keys:

- `customers.Customer_ID`
- `products.Product_ID`
- `orders.Order_ID`
- `order_items.Order_Item_ID`
- `returns.Return_ID`
- `marketing.Campaign_ID`

In [30]:
primary_keys = {
    "customers": "Customer_ID",
    "products": "Product_ID",
    "orders": "Order_ID",
    "order_items": "Order_Item_ID",
    "returns": "Return_ID",
    "marketing": "Campaign_ID"
}

pk_report = []

for name, key in primary_keys.items():
    df = datasets[name]

    pk_report.append({
        "Dataset": name,
        "Primary_Key": key,
        "Rows": len(df),
        "Unique_Keys": df[key].nunique(dropna=True),
        "Missing_Keys": int(df[key].isna().sum()),
        "Duplicate_Keys": int(df[key].duplicated().sum())
    })

pd.DataFrame(pk_report)

,Dataset,Primary_Key,Rows,Unique_Keys,Missing_Keys,Duplicate_Keys
0,customers,Customer_ID,25000,25000,0,0
1,products,Product_ID,1200,1200,0,0
2,orders,Order_ID,112000,112000,0,0
3,order_items,Order_Item_ID,227961,227511,0,450
4,returns,Return_ID,16185,16185,0,0
5,marketing,Campaign_ID,360,360,0,0


## 10. Invalid date audit

We use `errors="coerce"` so invalid strings become `NaT`.

Do not clean them yet — this section is only identifying problems.

In [31]:
date_columns = {
    "customers": ["Signup_Date"],
    "orders": ["Order_Date", "Ship_Date"],
    "returns": ["Return_Date"],
    "marketing": ["Month"]
}

invalid_dates = []

for name, cols in date_columns.items():
    df = datasets[name]

    for col in cols:
        converted = pd.to_datetime(df[col], errors="coerce")

        # Existing missing values should not be counted as invalid strings.
        invalid_mask = converted.isna() & df[col].notna()

        invalid_dates.append({
            "Dataset": name,
            "Column": col,
            "Invalid_Date_Values": int(invalid_mask.sum())
        })

invalid_dates_df = pd.DataFrame(invalid_dates)
invalid_dates_df

,Dataset,Column,Invalid_Date_Values
0,customers,Signup_Date,0
1,orders,Order_Date,100
2,orders,Ship_Date,150
3,returns,Return_Date,100
4,marketing,Month,0


### View the actual invalid date values

In [32]:
for name, cols in date_columns.items():
    df = datasets[name]

    for col in cols:
        converted = pd.to_datetime(df[col], errors="coerce")
        invalid_mask = converted.isna() & df[col].notna()

        if invalid_mask.any():
            print(f"\n{name}.{col}")
            print(df.loc[invalid_mask, col].value_counts().head(20))


orders.Order_Date
Order_Date
2025-13-05    100
Name: count, dtype: int64

orders.Ship_Date
Ship_Date
not available    150
Name: count, dtype: int64

returns.Return_Date
Return_Date
invalid date    100
Name: count, dtype: int64


## 11. Numeric-value audit

We will look for impossible values such as:

- quantity <= 0
- price <= 0
- negative revenue
- discount outside 0–100%
- negative cost
- negative refund
- negative marketing spend
- clicks greater than impressions

In [33]:
numeric_issues = []

def add_issue(dataset, rule, count):
    numeric_issues.append({
        "Dataset": dataset,
        "Rule": rule,
        "Issue_Count": int(count)
    })

add_issue("order_items", "Quantity <= 0", (order_items["Quantity"] <= 0).sum())
add_issue("order_items", "Unit_Price <= 0", (order_items["Unit_Price"] <= 0).sum())
add_issue("order_items", "Gross_Sales < 0", (order_items["Gross_Sales"] < 0).sum())
add_issue("order_items", "Net_Sales < 0", (order_items["Net_Sales"] < 0).sum())
add_issue("order_items", "Cost_Amount < 0", (order_items["Cost_Amount"] < 0).sum())
add_issue("order_items", "Discount_Pct < 0 or > 1",
          ((order_items["Discount_Pct"] < 0) | (order_items["Discount_Pct"] > 1)).sum())

add_issue("products", "List_Price <= 0", (products["List_Price"] <= 0).sum())
add_issue("products", "Unit_Cost <= 0", (products["Unit_Cost"] <= 0).sum())

add_issue("returns", "Refund_Amount < 0", (returns["Refund_Amount"] < 0).sum())

add_issue("marketing", "Spend < 0", (marketing["Spend"] < 0).sum())
add_issue("marketing", "Clicks < 0", (marketing["Clicks"] < 0).sum())
add_issue("marketing", "Impressions < 0", (marketing["Impressions"] < 0).sum())
add_issue("marketing", "Clicks > Impressions",
          (marketing["Clicks"] > marketing["Impressions"]).sum())
add_issue("marketing", "Attributed_Orders > Clicks",
          (marketing["Attributed_Orders"] > marketing["Clicks"]).sum())

numeric_issues_df = pd.DataFrame(numeric_issues)
numeric_issues_df

,Dataset,Rule,Issue_Count
0,order_items,Quantity <= 0,150
1,order_items,Unit_Price <= 0,100
2,order_items,Gross_Sales < 0,0
3,order_items,Net_Sales < 0,0
4,order_items,Cost_Amount < 0,0
5,order_items,Discount_Pct < 0 or > 1,0
6,products,List_Price <= 0,0
7,products,Unit_Cost <= 0,0
8,returns,Refund_Amount < 0,0
9,marketing,Spend < 0,0


## 12. Text/category consistency

This is one of the most important checks for Power BI because inconsistent labels create duplicate categories in visuals.

We'll inspect common business dimensions.

In [34]:
categorical_columns = {
    "customers": ["City", "State", "Customer_Segment", "Email_Opt_In"],
    "products": ["Category", "Subcategory", "Brand"],
    "orders": ["Channel", "Payment_Method", "Shipping_Method", "Order_Status"],
    "returns": ["Return_Reason", "Return_Status"],
    "marketing": ["Channel", "Campaign_Type"]
}

for name, cols in categorical_columns.items():
    df = datasets[name]

    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)

    for col in cols:
        print(f"\n--- {col} ---")
        print(df[col].value_counts(dropna=False).head(30))


CUSTOMERS

--- City ---
City
Other           7370
New York        2304
Los Angeles     1890
Chicago         1311
Houston         1255
Phoenix          786
Philadelphia     786
Dallas           766
San Antonio      697
Miami            692
Denver           688
Atlanta          682
Austin           649
Boston           642
San Diego        633
Seattle          619
Charlotte        580
Orlando          565
Portland         518
Columbus         506
Nashville        486
OTHER             76
 Other            73
New Yrok          25
Huston            25
Seatle            25
Los Angles        25
Chcago            25
 New York         22
NEW YORK          22
Name: count, dtype: int64

--- State ---
State
Other    7519
TX       3446
CA       2592
NY       2373
IL       1355
FL       1278
AZ        806
PA        801
CO        702
GA        693
WA        654
MA        650
NC        588
OR        530
OH        515
TN        498
Name: count, dtype: int64

--- Customer_Segment ---
Customer_Segment


## 13. Leading/trailing whitespace audit

In [35]:
whitespace_report = []

for name, df in datasets.items():
    for col in df.select_dtypes(include="object").columns:
        s = df[col].dropna().astype(str)
        whitespace_count = (s != s.str.strip()).sum()

        if whitespace_count > 0:
            whitespace_report.append({
                "Dataset": name,
                "Column": col,
                "Whitespace_Issues": int(whitespace_count)
            })

whitespace_df = pd.DataFrame(whitespace_report)
whitespace_df

C:\Users\Didar\AppData\Local\Temp\ipykernel_22596\4262812120.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:
C:\Users\Didar\AppData\Local\Temp\ipykernel_22596\4262812120.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

,Dataset,Column,Whitespace_Issues
0,customers,City,200
1,products,Category,30
2,orders,Payment_Method,200
3,returns,Return_Reason,80


## 14. Foreign-key integrity

Relationships expected:

- `orders.Customer_ID` → `customers.Customer_ID`
- `order_items.Order_ID` → `orders.Order_ID`
- `order_items.Product_ID` → `products.Product_ID`
- `returns.Order_Item_ID` → `order_items.Order_Item_ID`
- `returns.Order_ID` → `orders.Order_ID`
- `returns.Product_ID` → `products.Product_ID`

In [36]:
fk_report = []

def check_fk(child_name, child_df, child_col, parent_name, parent_df, parent_col):
    missing_refs = (
        child_df[child_col]
        .dropna()
        .isin(parent_df[parent_col])
        .eq(False)
        .sum()
    )

    fk_report.append({
        "Child_Table": child_name,
        "Child_Key": child_col,
        "Parent_Table": parent_name,
        "Parent_Key": parent_col,
        "Missing_References": int(missing_refs)
    })

check_fk("orders", orders, "Customer_ID",
         "customers", customers, "Customer_ID")

check_fk("order_items", order_items, "Order_ID",
         "orders", orders, "Order_ID")

check_fk("order_items", order_items, "Product_ID",
         "products", products, "Product_ID")

check_fk("returns", returns, "Order_Item_ID",
         "order_items", order_items, "Order_Item_ID")

check_fk("returns", returns, "Order_ID",
         "orders", orders, "Order_ID")

check_fk("returns", returns, "Product_ID",
         "products", products, "Product_ID")

fk_df = pd.DataFrame(fk_report)
fk_df

,Child_Table,Child_Key,Parent_Table,Parent_Key,Missing_References
0,orders,Customer_ID,customers,Customer_ID,0
1,order_items,Order_ID,orders,Order_ID,0
2,order_items,Product_ID,products,Product_ID,0
3,returns,Order_Item_ID,order_items,Order_Item_ID,0
4,returns,Order_ID,orders,Order_ID,0
5,returns,Product_ID,products,Product_ID,0


## 15. Cross-table business-rule checks

These checks go beyond simple nulls and duplicates.

In [37]:
business_rule_issues = []

def add_business_issue(rule, count):
    business_rule_issues.append({
        "Rule": rule,
        "Issue_Count": int(count)
    })

# Prepare valid dates safely.
orders_check = orders.copy()
orders_check["Order_Date_Parsed"] = pd.to_datetime(
    orders_check["Order_Date"], errors="coerce"
)
orders_check["Ship_Date_Parsed"] = pd.to_datetime(
    orders_check["Ship_Date"], errors="coerce"
)

# Ship date before order date
mask = (
    orders_check["Ship_Date_Parsed"].notna()
    & orders_check["Order_Date_Parsed"].notna()
    & (orders_check["Ship_Date_Parsed"] < orders_check["Order_Date_Parsed"])
)
add_business_issue("Ship_Date earlier than Order_Date", mask.sum())

# Completed order missing ship date
mask = (
    orders["Order_Status"].eq("Completed")
    & orders_check["Ship_Date_Parsed"].isna()
)
add_business_issue("Completed order missing/invalid Ship_Date", mask.sum())

# Cancelled order with recognized sales
cancelled_ids = set(
    orders.loc[orders["Order_Status"].eq("Cancelled"), "Order_ID"]
)

mask = (
    order_items["Order_ID"].isin(cancelled_ids)
    & (order_items["Net_Sales"] != 0)
)
add_business_issue("Cancelled order item with non-zero Net_Sales", mask.sum())

# Profit mathematical check
expected_profit = (
    order_items["Net_Sales"] - order_items["Cost_Amount"]
)

mask = ~np.isclose(
    order_items["Profit"].fillna(np.nan),
    expected_profit.fillna(np.nan),
    atol=0.02,
    equal_nan=True
)

add_business_issue("Profit != Net_Sales - Cost_Amount", mask.sum())

# Gross sales mathematical check
expected_gross = (
    order_items["Quantity"] * order_items["Unit_Price"]
)

mask = ~np.isclose(
    order_items["Gross_Sales"].fillna(np.nan),
    expected_gross.fillna(np.nan),
    atol=0.02,
    equal_nan=True
)

add_business_issue("Gross_Sales != Quantity × Unit_Price", mask.sum())

business_rules_df = pd.DataFrame(business_rule_issues)
business_rules_df

,Rule,Issue_Count
0,Ship_Date earlier than Order_Date,0
1,Completed order missing/invalid Ship_Date,146
2,Cancelled order item with non-zero Net_Sales,0
3,Profit != Net_Sales - Cost_Amount,0
4,Gross_Sales != Quantity × Unit_Price,250


## 16. Inspect suspected duplicate order items

The raw dataset intentionally contains some duplicate transaction lines.

In [38]:
duplicate_items = order_items[
    order_items.duplicated(keep=False)
].sort_values("Order_ID")

print("Duplicate order-item rows:", len(duplicate_items))
display(duplicate_items.head(20))

Duplicate order-item rows: 900


,Order_Item_ID,Order_ID,Product_ID,Quantity,Unit_Price,Discount_Pct,Gross_Sales,Discount_Amount,Net_Sales,Cost_Amount,Profit
763,OI00000764,O0000379,P00756,1,59.42,0.05,59.42,2.97,56.45,41.48,14.97
227773,OI00000764,O0000379,P00756,1,59.42,0.05,59.42,2.97,56.45,41.48,14.97
1552,OI00001553,O0000755,P00372,1,37.82,0.00,37.82,0.00,37.82,22.27,15.55
227844,OI00001553,O0000755,P00372,1,37.82,0.00,37.82,0.00,37.82,22.27,15.55
1733,OI00001734,O0000845,P00914,3,25.09,0.10,75.27,7.53,67.74,48.09,19.65
227737,OI00001734,O0000845,P00914,3,25.09,0.10,75.27,7.53,67.74,48.09,19.65
1895,OI00001896,O0000921,P00807,1,50.46,0.10,50.46,5.05,45.41,27.97,17.44
227870,OI00001896,O0000921,P00807,1,50.46,0.10,50.46,5.05,45.41,27.97,17.44
3636,OI00003637,O0001786,P00807,1,50.46,0.00,50.46,0.00,50.46,27.97,22.49
227664,OI00003637,O0001786,P00807,1,50.46,0.00,50.46,0.00,50.46,27.97,22.49


## 17. Quick statistical summary

In [39]:
print("ORDER ITEMS")
display(
    order_items[
        [
            "Quantity",
            "Unit_Price",
            "Discount_Pct",
            "Gross_Sales",
            "Net_Sales",
            "Cost_Amount",
            "Profit"
        ]
    ].describe()
)

print("\nPRODUCTS")
display(
    products[
        [
            "List_Price",
            "Unit_Cost",
            "Base_Margin",
            "Base_Return_Rate"
        ]
    ].describe()
)

print("\nMARKETING")
display(
    marketing[
        [
            "Spend",
            "Impressions",
            "Clicks",
            "Attributed_Orders",
            "CTR",
            "Conversion_Rate"
        ]
    ].describe()
)

ORDER ITEMS


,Quantity,Unit_Price,Discount_Pct,Gross_Sales,Net_Sales,Cost_Amount,Profit
count,"227,961.00","227,961.00","227,661.00","227,961.00","227,961.00","227,961.00","227,961.00"
mean,1.45,55.41,0.06,80.42,72.41,50.70,21.70
std,0.80,35.70,0.06,73.93,69.93,52.58,21.09
min,-1.00,0.00,0.00,7.45,0.00,0.00,-24.12
25%,1.00,32.64,0.00,37.23,32.72,21.23,9.56
50%,1.00,47.32,0.05,58.21,52.80,36.04,15.97
75%,2.00,66.38,0.10,97.74,88.96,61.77,26.40
max,6.00,280.00,0.30,"1,560.72","1,560.72","1,050.96",509.76



PRODUCTS


,List_Price,Unit_Cost,Base_Margin,Base_Return_Rate
count,"1,200.00","1,200.00","1,200.00","1,200.00"
mean,55.80,36.70,0.36,0.07
std,37.11,27.73,0.10,0.03
min,7.45,3.89,0.18,0.04
25%,31.18,18.52,0.29,0.04
50%,46.84,28.83,0.35,0.07
75%,68.50,45.46,0.42,0.09
max,280.00,227.59,0.62,0.13



MARKETING


,Spend,Impressions,Clicks,Attributed_Orders,CTR,Conversion_Rate
count,350.00,360.00,350.00,360.00,360.00,360.00
mean,"9,381.24","625,783.66","20,710.37",943.38,0.03,0.04
std,"7,327.84","477,071.60","18,212.94","1,062.71",0.01,0.01
min,"1,598.78","88,821.00","2,256.00",100.00,0.02,0.02
25%,"4,002.78","271,899.25","8,296.50",319.25,0.03,0.04
50%,"6,955.58","487,082.50","15,157.00",519.50,0.03,0.04
75%,"12,733.45","825,219.50","28,702.00","1,175.75",0.04,0.05
max,"40,002.18","2,420,694.00","98,828.00","6,247.00",0.06,0.06


## 18. Build final data-quality summary

In [40]:
summary_rows = []

summary_rows.append({
    "Check": "Missing values",
    "Issues_Found": int(missing_df["Missing_Count"].sum()) if len(missing_df) else 0
})

summary_rows.append({
    "Check": "Exact duplicate rows",
    "Issues_Found": int(duplicate_df["Exact_Duplicate_Rows"].sum())
})

summary_rows.append({
    "Check": "Duplicate primary keys",
    "Issues_Found": int(pd.DataFrame(pk_report)["Duplicate_Keys"].sum())
})

summary_rows.append({
    "Check": "Invalid date values",
    "Issues_Found": int(invalid_dates_df["Invalid_Date_Values"].sum())
})

summary_rows.append({
    "Check": "Numeric-rule violations",
    "Issues_Found": int(numeric_issues_df["Issue_Count"].sum())
})

summary_rows.append({
    "Check": "Whitespace issues",
    "Issues_Found": int(whitespace_df["Whitespace_Issues"].sum()) if len(whitespace_df) else 0
})

summary_rows.append({
    "Check": "Foreign-key violations",
    "Issues_Found": int(fk_df["Missing_References"].sum())
})

summary_rows.append({
    "Check": "Business-rule violations",
    "Issues_Found": int(business_rules_df["Issue_Count"].sum())
})

quality_summary = pd.DataFrame(summary_rows)
quality_summary

,Check,Issues_Found
0,Missing values,5770
1,Exact duplicate rows,450
2,Duplicate primary keys,450
3,Invalid date values,350
4,Numeric-rule violations,250
5,Whitespace issues,510
6,Foreign-key violations,0
7,Business-rule violations,396


## 19. Audit conclusion

At this stage, **do not fix anything yet**.

The purpose of this notebook is to create a documented baseline of the problems found in the raw client data.

Typical issues expected in this project include:

- missing values
- duplicate transaction rows
- invalid dates
- inconsistent capitalization
- leading/trailing spaces
- city misspellings
- negative quantities
- zero unit prices
- missing product IDs
- inconsistent payment/channel labels
- inconsistent return reasons/status
- missing marketing metrics
